In [10]:
import os, json, hashlib, platform
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

DATA_FILE = 'loan.csv'
DICT_FILE = 'LCDataDictionary.xlsx'
OUT = 'acrs_data'
SEED = 42
STRICT_DEFAULT = True   # True: Charged Off/Default vs Fully Paid. False: the old definition (late loans = default)
os.makedirs(OUT, exist_ok=True)

# Exact environment, for thesis section 3.4.1, the README and requirements.txt
import sklearn, xgboost, lightgbm, shap, optuna, scipy
env = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__, 'lightgbm': lightgbm.__version__,
    'shap': shap.__version__, 'optuna': optuna.__version__, 'scipy': scipy.__version__,
    'platform': platform.platform(), 'cpu_count': os.cpu_count(),
    'seed': SEED, 'strict_default': STRICT_DEFAULT,
}
sha = hashlib.sha256()
with open(DATA_FILE, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        sha.update(chunk)
env['data_file'] = DATA_FILE
env['data_sha256'] = sha.hexdigest()
env['data_bytes'] = os.path.getsize(DATA_FILE)
print(json.dumps(env, indent=2))

{
  "python": "3.12.7",
  "numpy": "1.26.4",
  "pandas": "2.3.1",
  "scikit-learn": "1.7.1",
  "xgboost": "3.4.1",
  "lightgbm": "4.7.0",
  "shap": "0.52.0",
  "optuna": "5.0.0",
  "scipy": "1.16.0",
  "platform": "Windows-11-10.0.26200-SP0",
  "cpu_count": 12,
  "seed": 42,
  "strict_default": true,
  "data_file": "loan.csv",
  "data_sha256": "23783ef320e4df24ac113d6e5b830edb909912b7783d49b89aacd5690dc9120c",
  "data_bytes": 1189395649
}


## 1. Load the raw file
`issue_d` is parsed as a real date. (Sorting it as text made "Sep-2018" look like the last month; the data actually run to December 2018.)

In [12]:
df = pd.read_csv(DATA_FILE, low_memory=False)
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['year'] = df['issue_d'].dt.year
env['raw_rows'], env['raw_cols'] = int(df.shape[0]), int(df.shape[1])
env['issue_first'] = df['issue_d'].min().strftime('%b %Y')
env['issue_last'] = df['issue_d'].max().strftime('%b %Y')
print(f"Raw shape: {df.shape}")
print(f"Issue dates: {env['issue_first']} to {env['issue_last']}")
print(df['loan_status'].value_counts())

Raw shape: (2260668, 146)
Issue dates: Jun 2007 to Dec 2018
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


## 2. Default definition and loan-maturity bias
Only loans with a final outcome can be labelled. Recent vintages have fewer resolved loans, and the ones that are resolved
are biased towards early defaults and early repayments. This table quantifies that for the thesis (section 4.2.1).

In [14]:
FINAL = ['Fully Paid', 'Charged Off', 'Default']
LATE = ['Late (31-120 days)', 'Late (16-30 days)']
POSITIVE = ['Charged Off', 'Default'] + ([] if STRICT_DEFAULT else LATE)
KEEP = ['Fully Paid'] + POSITIVE

by_year = df.groupby('year').agg(
    loans=('loan_status', 'size'),
    resolved_share=('loan_status', lambda s: s.isin(FINAL).mean()),
    late_share=('loan_status', lambda s: s.isin(LATE).mean()),
    current_share=('loan_status', lambda s: s.isin(['Current', 'In Grace Period']).mean()),
)
kept = df[df['loan_status'].isin(KEEP)]
by_year['modelled_loans'] = kept.groupby('year').size()
by_year['default_rate'] = kept.groupby('year')['loan_status'].apply(lambda s: s.isin(POSITIVE).mean())
by_year['term60_share'] = kept.groupby('year')['term'].apply(lambda s: s.astype(str).str.contains('60').mean())
by_year.round(4).to_csv(f'{OUT}/resolved_share_by_year.csv')
print(by_year.round(3).to_string())

       loans  resolved_share  late_share  current_share  modelled_loans  default_rate  term60_share
year                                                                                               
2007     603           0.416       0.000          0.000             251         0.179         0.000
2008    2393           0.653       0.000          0.000            1562         0.158         0.000
2009    5281           0.893       0.000          0.000            4716         0.126         0.000
2010   12537           0.920       0.000          0.000           11536         0.129         0.266
2011   21721           1.000       0.000          0.000           21721         0.152         0.351
2012   53367           1.000       0.000          0.000           53367         0.162         0.185
2013  134814           1.000       0.000          0.000          134793         0.156         0.255
2014  235629           0.940       0.002          0.058          221468         0.185         0.266


## 3. Label, sort by issue date, split by time
Train 2007–2014, validation 2015, test 2016–2018. Loans are sorted by issue date so the test stream is chronological.

In [16]:
df = df[df['loan_status'].isin(KEEP)].copy()
df['target'] = df['loan_status'].isin(POSITIVE).astype('int8')
df = df.sort_values('issue_d', kind='stable').reset_index(drop=True)

train = df[df['year'] <= 2014].copy()
val = df[df['year'] == 2015].copy()
test = df[df['year'] >= 2016].copy()

env['modelled_rows'] = int(len(df))
env['partitions'] = {}
for name, part in [('train', train), ('val', val), ('test', test)]:
    info = {'rows': int(len(part)), 'default_rate': round(float(part['target'].mean()), 4),
            'first': part['issue_d'].min().strftime('%b %Y'), 'last': part['issue_d'].max().strftime('%b %Y')}
    env['partitions'][name] = info
    print(f"{name:<5}: {info['rows']:>9,} loans | default rate {info['default_rate']:.3f} | {info['first']} to {info['last']}")

train:   449,414 loans | default rate 0.170 | Jun 2007 to Dec 2014
val  :   373,412 loans | default rate 0.202 | Jan 2015 to Dec 2015
test :   480,812 loans | default rate 0.229 | Jan 2016 to Dec 2018


## 4. Decide the feature columns — using the training period only
* **Post-origination (leakage):** values that exist only after the loan is issued.
* **Redundant:** known at origination but a function of other features (`installment` = f(amount, rate, term); `funded_amnt_inv` ≈ `funded_amnt`).
* **High missing:** more than 40% missing **in the training period**.

In [18]:
POST_ORIGINATION = [
    # payments and balances after issuance
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv',
    # dates after issuance
    'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d',
    # status flags that Lending Club sets only after delinquency or charge-off
    'debt_settlement_flag', 'hardship_flag', 'pymnt_plan',
]
POST_ORIGINATION += [c for c in df.columns
                     if c.startswith(('hardship_', 'settlement_', 'debt_settlement_flag_', 'last_fico'))
                     or c in ('deferral_term', 'payment_plan_start_date', 'orig_projected_additional_accrued_interest')]
POST_ORIGINATION = [c for c in dict.fromkeys(POST_ORIGINATION) if c in df.columns]
REDUNDANT = [c for c in ['installment', 'funded_amnt_inv'] if c in df.columns]
NOT_FEATURES = ['id', 'member_id', 'url', 'desc', 'title', 'zip_code', 'emp_title', 'loan_status',
                'issue_d', 'year', 'target', 'addr_state', 'earliest_cr_line']

missing_train = train.isnull().mean()
candidates = [c for c in df.columns if c not in NOT_FEATURES + POST_ORIGINATION + REDUNDANT]
HIGH_MISSING = [c for c in candidates if missing_train[c] > 0.40]
CONSTANT = [c for c in candidates if c not in HIGH_MISSING and train[c].nunique(dropna=True) <= 1]
FEATURES_RAW = [c for c in candidates if c not in HIGH_MISSING + CONSTANT]

# The leaky diagnostic model also gets the post-origination and redundant columns (except dates)
DIAG_EXTRA = [c for c in POST_ORIGINATION + REDUNDANT
              if not c.endswith(('_d', '_date')) and missing_train[c] <= 0.40 and train[c].nunique(dropna=True) > 1]
DIAG_RAW = FEATURES_RAW + DIAG_EXTRA

suspicious = [c for c in FEATURES_RAW
              if any(k in c for k in ('pymnt', 'total_rec', 'recover', 'settle', 'hardship'))
              or c.startswith(('last_', 'next_'))]
print(f"Post-origination columns removed ({len(POST_ORIGINATION)}): {POST_ORIGINATION}")
print(f"Redundant columns removed: {REDUNDANT}")
print(f"High-missing columns removed (>40% missing in training): {len(HIGH_MISSING)}")
print(f"Constant columns removed: {CONSTANT}")
print(f"Raw feature columns kept: {len(FEATURES_RAW)}")
print(f"Check by hand — names that look post-origination: {suspicious if suspicious else 'none'}")

Post-origination columns removed (36): ['total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d', 'debt_settlement_flag', 'hardship_flag', 'pymnt_plan', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term']
Redundant columns removed: ['installment', 'funded_amnt_inv']
High-missing columns removed (>40% missing in training): 33
Constant columns removed: ['policy_code', 'application_type', 'disbursement_meth

## 5. Encode and impute — every statistic fitted on training only 
Ordinal: grade, sub_grade, emp_length, term. Numeric gaps: training median. Categorical gaps: training mode.
One-hot levels: the training categories (a category seen only later becomes the reference level).

In [20]:
GRADE = {g: i + 1 for i, g in enumerate('ABCDEFG')}
SUB_GRADE = {f'{g}{n}': i + 1 for i, (g, n) in enumerate((g, n) for g in 'ABCDEFG' for n in range(1, 6))}
EMP = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4, '5 years': 5,
       '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}

def ordinal(X):
    X = X.copy()
    if 'grade' in X: X['grade'] = X['grade'].map(GRADE)
    if 'sub_grade' in X: X['sub_grade'] = X['sub_grade'].map(SUB_GRADE)
    if 'emp_length' in X: X['emp_length'] = X['emp_length'].map(EMP)
    if 'term' in X: X['term'] = pd.to_numeric(X['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
    return X

def prepare(part, cols, fit=None):
    """Ordinal-encode, impute and one-hot encode. fit=None means: learn the statistics from this part (training)."""
    X = ordinal(part[cols])
    if fit is None:
        num = X.select_dtypes(include='number').columns.tolist()
        cat = [c for c in X.columns if c not in num]
        fit = {'num': num, 'cat': cat, 'median': X[num].median(),
               'mode': {c: (X[c].mode().iloc[0] if X[c].notna().any() else 'Unknown') for c in cat}}
        fit['levels'] = {c: sorted(X[c].fillna(fit['mode'][c]).astype(str).unique()) for c in cat}
    X[fit['num']] = X[fit['num']].fillna(fit['median']).fillna(0)
    blocks = [X[fit['num']].astype('float64')]
    for c in fit['cat']:
        values = X[c].fillna(fit['mode'][c]).astype(str)
        levels = fit['levels'][c][1:]           # first level = reference (drop_first)
        blocks.append(pd.DataFrame({f'{c}_{lv}': (values == lv).astype('uint8') for lv in levels}, index=X.index))
    return pd.concat(blocks, axis=1), fit

X_train_raw, fit_clean = prepare(train, FEATURES_RAW)
X_val_raw, _ = prepare(val, FEATURES_RAW, fit_clean)
X_test_raw, _ = prepare(test, FEATURES_RAW, fit_clean)
y_train = train['target'].to_numpy()
y_val = val['target'].to_numpy()
y_test = test['target'].to_numpy()
print(f"Encoded features: {X_train_raw.shape[1]}")
print(f"Categorical columns one-hot encoded: {fit_clean['cat']}")

Encoded features: 77
Categorical columns one-hot encoded: ['home_ownership', 'verification_status', 'purpose', 'initial_list_status']


## 6. Leakage diagnostic: like-for-like before/after
Same quick XGBoost (100 trees, depth 5), same training data, same **test** set — once with the leakage columns, once without.

In [22]:
Xd_train, fit_diag = prepare(train, DIAG_RAW)
Xd_val, _ = prepare(val, DIAG_RAW, fit_diag)
Xd_test, _ = prepare(test, DIAG_RAW, fit_diag)
DIAG = dict(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=SEED, n_jobs=-1, eval_metric='auc',
            enable_categorical=False)

diag = {'model': 'XGBoost, 100 trees, depth 5, learning rate 0.1', 'leakage_columns_encoded':
        [c for c in Xd_train.columns if c not in X_train_raw.columns]}
for label, (Xtr, Xva, Xte) in {'with_leakage': (Xd_train, Xd_val, Xd_test),
                               'without_leakage': (X_train_raw, X_val_raw, X_test_raw)}.items():
    m = XGBClassifier(**DIAG).fit(Xtr, y_train)
    diag[label] = {'features': int(Xtr.shape[1]),
                   'val_auc': round(roc_auc_score(y_val, m.predict_proba(Xva)[:, 1]), 4),
                   'test_auc': round(roc_auc_score(y_test, m.predict_proba(Xte)[:, 1]), 4)}
    print(f"{label:<16} features={diag[label]['features']:>3} | val ROC-AUC={diag[label]['val_auc']:.4f} | test ROC-AUC={diag[label]['test_auc']:.4f}")
with open(f'{OUT}/leakage_diagnostic.json', 'w') as f:
    json.dump(diag, f, indent=2)

with_leakage     features= 88 | val ROC-AUC=0.9998 | test ROC-AUC=0.9999
without_leakage  features= 77 | val ROC-AUC=0.7400 | test ROC-AUC=0.7101


## 7. Scale (fitted on training only) and save

In [24]:
scaler = MinMaxScaler().fit(X_train_raw)
def scaled(X):
    return pd.DataFrame(scaler.transform(X), columns=X_train_raw.columns)

X_train, X_val, X_test = scaled(X_train_raw), scaled(X_val_raw), scaled(X_test_raw)
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    X.to_parquet(f'{OUT}/X_{name}.parquet')
    np.save(f'{OUT}/y_{name}.npy', y)

# Per-loan information used only for evaluation (never as a model input)
META_COLS = ['issue_d', 'loan_status', 'term', 'grade', 'sub_grade', 'funded_amnt', 'int_rate', 'total_pymnt']
for name, part in [('train', train), ('val', val), ('test', test)]:
    meta = part[[c for c in META_COLS if c in part.columns]].reset_index(drop=True)
    meta['term_months'] = pd.to_numeric(meta['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
    meta.to_parquet(f'{OUT}/meta_{name}.parquet')

lists = {'post_origination_removed': POST_ORIGINATION, 'redundant_removed': REDUNDANT,
         'high_missing_removed': HIGH_MISSING, 'constant_removed': CONSTANT,
         'raw_features': FEATURES_RAW, 'encoded_features': list(X_train.columns)}
with open(f'{OUT}/feature_lists.json', 'w') as f:
    json.dump(lists, f, indent=2)
print(f"Saved X/y/meta for train, val, test with {X_train.shape[1]} features")

Saved X/y/meta for train, val, test with 77 features


## 8. Feature list for the appendix

In [26]:
try:
    dd = pd.read_excel(DICT_FILE, sheet_name='LoanStats').iloc[:, :2]
    dd.columns = ['name', 'description']
    dd['name'] = dd['name'].astype(str).str.replace('\xa0', ' ').str.strip()
    desc = dict(zip(dd['name'], dd['description']))
except Exception as e:
    print(f"Data dictionary not read ({e}); descriptions left empty")
    desc = {}
rows = []
for feat in X_train.columns:
    source = feat if feat in fit_clean['num'] else next((c for c in fit_clean['cat'] if feat.startswith(c + '_')), feat)
    rows.append({'feature': feat, 'source_column': source,
                 'encoding': 'one-hot' if source != feat else ('ordinal' if source in ('grade', 'sub_grade', 'emp_length', 'term') else 'numeric'),
                 'description': desc.get(source, '')})
pd.DataFrame(rows).to_csv(f'{OUT}/feature_dictionary.csv', index=False)
print(f"Saved feature_dictionary.csv ({len(rows)} rows) — mark each as 'application' or 'credit report at origination' for the appendix")

Saved feature_dictionary.csv (77 rows) — mark each as 'application' or 'credit report at origination' for the appendix


## 9. Verification

In [28]:
BANNED = ['total_pymnt', 'total_rec', 'recoveries', 'collection_recovery', 'last_pymnt', 'out_prncp',
          'debt_settlement', 'hardship', 'pymnt_plan', 'installment', 'funded_amnt_inv', 'last_credit_pull', 'next_pymnt']
leftover = [c for c in X_train.columns if any(b in c for b in BANNED)]
assert not leftover, f"Leakage columns still present: {leftover}"
assert X_train.isnull().sum().sum() == 0 and X_val.isnull().sum().sum() == 0 and X_test.isnull().sum().sum() == 0
env['n_features'] = int(X_train.shape[1])
with open(f'{OUT}/environment.json', 'w') as f:
    json.dump(env, f, indent=2)
print("=" * 60)
print("PHASE 1 COMPLETE")
print(f"  Features: {X_train.shape[1]} (no leakage columns, no missing values)")
for name in ['train', 'val', 'test']:
    p = env['partitions'][name]
    print(f"  {name:<5} {p['rows']:>9,} loans, default rate {p['default_rate']:.3f}, {p['first']} to {p['last']}")
print(f"  Leakage diagnostic test ROC-AUC: {diag['with_leakage']['test_auc']} with leakage -> {diag['without_leakage']['test_auc']} without")
print("  Saved: environment.json, resolved_share_by_year.csv, leakage_diagnostic.json, feature_lists.json, feature_dictionary.csv")
print("=" * 60)

PHASE 1 COMPLETE
  Features: 77 (no leakage columns, no missing values)
  train   449,414 loans, default rate 0.170, Jun 2007 to Dec 2014
  val     373,412 loans, default rate 0.202, Jan 2015 to Dec 2015
  test    480,812 loans, default rate 0.229, Jan 2016 to Dec 2018
  Leakage diagnostic test ROC-AUC: 0.9999 with leakage -> 0.7101 without
  Saved: environment.json, resolved_share_by_year.csv, leakage_diagnostic.json, feature_lists.json, feature_dictionary.csv
